# 03. Strict QC Retrofit — Ligand State, Homology Reclustering, Leakage-Safe Split

**Inputs:** `data/manifest_qc_filtered_clustered_split.csv` (output of notebook 02).

**What this notebook does:**
1. Pulls non-polymer ligand info + R-free + mean B-factor for every PDB in the surviving manifest (one batched GraphQL trip, cached on disk).
2. Adds `ligand_match` flag per pair — was the meaningful ligand set the same in cryo and ambient? (Cryoprotectants/buffers/buffer ions are subtracted before comparison.)
3. Reclusters proteins by sequence using **MMseqs2** at 30% identity / 80% coverage if it's available; falls back to a tighter k-mer Jaccard otherwise.
4. Re-runs the cluster-aware train/val/test split with the new (homology-aware) clusters.
5. Writes two new manifests:
   - `data/manifest_v3.csv` — all 838 pairs, with new flags + new split.
   - `data/manifest_v3_strict.csv` — only pairs with `ligand_match=True` (the publication-quality subset).

**AdoMetDC** stays excluded throughout (held out for the case study).

> Re-running is fast — the GraphQL fetch caches to `data/cache/ligand_meta.json`.

## 1. Configuration

Tweak paths and thresholds here.

In [1]:
from pathlib import Path

# --- Inputs ---
DATA_DIR        = Path("data").resolve()
INPUT_MANIFEST  = DATA_DIR / "manifest_qc_filtered_clustered_split.csv"
CACHE_DIR       = DATA_DIR / "cache"
SEQ_DIR         = DATA_DIR / "sequences"
LIGAND_CACHE    = CACHE_DIR / "ligand_meta.json"

# --- Outputs ---
OUTPUT_FULL_MANIFEST   = DATA_DIR / "manifest_v3.csv"
OUTPUT_STRICT_MANIFEST = DATA_DIR / "manifest_v3_strict.csv"
SUMMARY_CSV            = DATA_DIR / "qc_v3_summary.csv"

CACHE_DIR.mkdir(exist_ok=True, parents=True)

# --- API ---
GRAPHQL_URL = "https://data.rcsb.org/graphql"

# --- Ligand denylist: cryoprotectants, buffers, common buffer ions ---
# Edit if your protein has unusual buffer components you want treated as meaningful.
CRYO_BUFFER_DENYLIST = {
    # Water + cryoprotectants + organic solvents
    "HOH", "DOD", "GOL", "EDO", "MPD", "1PE", "PG4", "P6G", "PEG",
    "PE4", "PE8", "PEU", "PGE",                                  # PGE = triethylene glycol (cryoprotectant)
    "DMS", "BME", "DTT", "TCE", "MOH", "EOH",
    "IPA", "ACT", "ACY",
    # Buffers
    "TRS", "BTB", "EPE", "MES", "BIS", "HEP", "CHE", "TAR", "CIT",
    "FMT", "IMD", "IMZ",
    # Precipitants / additives
    "PO4", "SO4", "MLA", "NO3", "FLC", "AZI", "SCN", "NH4",      # SCN = thiocyanate, NH4 = ammonium (almost always buffer)
    # Buffer ions usually unrelated to function
    "CL", "NA", "K", "BR", "IOD", "CS", "LI", "RB",
}
# Ions usually catalytic/structural -> KEPT in the meaningful set
CATALYTIC_IONS = {"MG", "MN", "CA", "ZN", "FE", "CU", "NI", "CO"}

# --- MMseqs2 clustering params (used if mmseqs is in PATH) ---
MMSEQS_MIN_SEQ_ID = 0.30   # 30% identity
MMSEQS_COVERAGE   = 0.80   # 80% bidirectional coverage
MMSEQS_COV_MODE   = 0      # bidirectional

# --- Jaccard fallback (used only if mmseqs unavailable) ---
JACCARD_K         = 4
JACCARD_THRESHOLD = 0.10

# --- Split ---
SPLIT_SEED   = 42
TRAIN_FRAC   = 0.70
VAL_FRAC     = 0.15
TEST_FRAC    = 0.15

# --- Politeness ---
GRAPHQL_BATCH   = 100
REQUEST_TIMEOUT = 60

print(f"Reading: {INPUT_MANIFEST}")
print(f"Will write:")
print(f"  {OUTPUT_FULL_MANIFEST}")
print(f"  {OUTPUT_STRICT_MANIFEST}")
print(f"  {SUMMARY_CSV}")

Reading: C:\Users\Jenitha Patel\OneDrive - The University of Chicago\S26\DL_genomics\final project\data\manifest_qc_filtered_clustered_split.csv
Will write:
  C:\Users\Jenitha Patel\OneDrive - The University of Chicago\S26\DL_genomics\final project\data\manifest_v3.csv
  C:\Users\Jenitha Patel\OneDrive - The University of Chicago\S26\DL_genomics\final project\data\manifest_v3_strict.csv
  C:\Users\Jenitha Patel\OneDrive - The University of Chicago\S26\DL_genomics\final project\data\qc_v3_summary.csv


## 2. Load the QC-filtered manifest from notebook 02

In [2]:
import json, time, sys, urllib.request, subprocess, shutil, tempfile, random
from collections import defaultdict
from pathlib import Path

import numpy as np
import pandas as pd
from tqdm.auto import tqdm

manifest = pd.read_csv(INPUT_MANIFEST)
print(f"Loaded {len(manifest):,} rows from notebook 02")
print(f"Unique UniProts:        {manifest['uniprot'].nunique():,}")
print(f"Unique seq_clusters v2: {manifest['seq_cluster'].nunique():,}")
print(f"Existing split counts:  {manifest['split'].value_counts().to_dict()}")
manifest.head(3)

Loaded 838 rows from notebook 02
Unique UniProts:        838
Unique seq_clusters v2: 809
Existing split counts:  {'train': 586, 'val': 128, 'test': 124}


c:\Users\Jenitha Patel\miniconda3\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


,pair_id,uniprot,pdb_cryo,chain_cryo,pdb_ambient,chain_ambient,space_group,cell_a_cryo,cell_b_cryo,cell_c_cryo,...,sequence_path,resolution_diff_abs,max_cell_angle_dev_deg,frac_modeled_both,passes_qc,qc_filter_reasons,seq_cluster,seq_len,split,pdb_pair_key
0,P94692_1KEK_2C3P,P94692,1KEK,A,2C3P,A,P 21 21 21,86.110,145.760,210.260,...,sequences\P94692.fasta,0.43,NaN,1.0,True,pass,cluster_0678,1231,train,1KEK_2C3P
1,P22983_2DIK_1DIK,P22983,2DIK,A,1DIK,A,P 1 2 1,89.800,58.800,102.000,...,sequences\P22983.fasta,0.20,NaN,1.0,True,pass,cluster_0474,874,train,1DIK_2DIK
2,Q54468_1C7T_1QBA,Q54468,1C7T,A,1QBA,A,P 21 21 2,109.174,99.416,86.556,...,sequences\Q54468.fasta,0.05,NaN,1.0,True,pass,cluster_0761,858,train,1C7T_1QBA


## 3. Fetch non-polymer entities + R-free + mean B-factor

One batched GraphQL call per ~100 PDB IDs. Cached on disk so re-runs are instant.

In [3]:
def post_json(url, payload, timeout=REQUEST_TIMEOUT):
    req = urllib.request.Request(
        url,
        data=json.dumps(payload).encode("utf-8"),
        headers={"Content-Type": "application/json"},
    )
    with urllib.request.urlopen(req, timeout=timeout) as r:
        return json.loads(r.read())


def chunks(seq, n):
    seq = list(seq)
    for i in range(0, len(seq), n):
        yield seq[i:i + n]


GRAPHQL_LIGAND_QUERY = """
query ($ids: [String!]!) {
  entries(entry_ids: $ids) {
    rcsb_id
    nonpolymer_entities {
      pdbx_entity_nonpoly { comp_id }
    }
    refine { ls_R_factor_R_free B_iso_mean }
  }
}
""".strip()


def fetch_ligand_meta(pdb_ids):
    cache = {}
    if LIGAND_CACHE.exists():
        with LIGAND_CACHE.open() as f:
            cache = json.load(f)
    todo = [pid for pid in pdb_ids if pid not in cache]
    if todo:
        for batch in tqdm(list(chunks(todo, GRAPHQL_BATCH)), desc="Fetching ligands+refine"):
            data = post_json(GRAPHQL_URL,
                             {"query": GRAPHQL_LIGAND_QUERY, "variables": {"ids": batch}})
            for entry in (data.get("data") or {}).get("entries") or []:
                if entry and entry.get("rcsb_id"):
                    cache[entry["rcsb_id"]] = entry
            with LIGAND_CACHE.open("w") as f:
                json.dump(cache, f)
            time.sleep(0.05)
    return cache


unique_pdb_ids = sorted(set(manifest["pdb_cryo"]) | set(manifest["pdb_ambient"]))
print(f"Unique PDB IDs to fetch: {len(unique_pdb_ids):,}")
ligand_meta = fetch_ligand_meta(unique_pdb_ids)
print(f"Got metadata for {len(ligand_meta):,} entries")

Unique PDB IDs to fetch: 1,674
Got metadata for 1,674 entries


## 4. Compute ligand sets and per-pair quality flags

`meaningful_ligands` = all non-polymer comp_ids minus the buffer/cryoprotectant denylist. Two structures "match" iff they share the same meaningful set (same substrates / cofactors).

In [4]:
def meaningful_ligands(entry):
    out = set()
    for npe in entry.get("nonpolymer_entities") or []:
        cid = ((npe.get("pdbx_entity_nonpoly") or {}).get("comp_id") or "").strip().upper()
        if not cid or cid in CRYO_BUFFER_DENYLIST:
            continue
        out.add(cid)
    return out


def r_free(entry):
    for r in entry.get("refine") or []:
        if r.get("ls_R_factor_R_free") is not None:
            return float(r["ls_R_factor_R_free"])
    return None


def b_iso_mean(entry):
    for r in entry.get("refine") or []:
        if r.get("B_iso_mean") is not None:
            return float(r["B_iso_mean"])
    return None


pdb_ligands = {pid: meaningful_ligands(e) for pid, e in ligand_meta.items()}
pdb_rfree   = {pid: r_free(e)             for pid, e in ligand_meta.items()}
pdb_biso    = {pid: b_iso_mean(e)         for pid, e in ligand_meta.items()}


def fmt_set(s):
    return ",".join(sorted(s)) if s else ""


df = manifest.copy()

df["ligands_cryo"]    = df["pdb_cryo"].map(lambda p: fmt_set(pdb_ligands.get(p, set())))
df["ligands_ambient"] = df["pdb_ambient"].map(lambda p: fmt_set(pdb_ligands.get(p, set())))
df["n_ligands_cryo"]    = df["pdb_cryo"].map(lambda p: len(pdb_ligands.get(p, set())))
df["n_ligands_ambient"] = df["pdb_ambient"].map(lambda p: len(pdb_ligands.get(p, set())))

df["ligand_match"] = df.apply(
    lambda r: pdb_ligands.get(r["pdb_cryo"], set()) == pdb_ligands.get(r["pdb_ambient"], set()),
    axis=1)
df["n_ligand_diff"] = df.apply(
    lambda r: len(pdb_ligands.get(r["pdb_cryo"], set()) ^ pdb_ligands.get(r["pdb_ambient"], set())),
    axis=1)
df["ligands_diff_set"] = df.apply(
    lambda r: fmt_set(pdb_ligands.get(r["pdb_cryo"], set()) ^ pdb_ligands.get(r["pdb_ambient"], set())),
    axis=1)

df["r_free_cryo"]      = df["pdb_cryo"].map(pdb_rfree)
df["r_free_ambient"]   = df["pdb_ambient"].map(pdb_rfree)
df["r_free_diff_abs"]  = (df["r_free_cryo"] - df["r_free_ambient"]).abs()
df["b_iso_mean_cryo"]    = df["pdb_cryo"].map(pdb_biso)
df["b_iso_mean_ambient"] = df["pdb_ambient"].map(pdb_biso)

n_match = int(df["ligand_match"].sum())
print(f"Pairs with matching ligand state:    {n_match:,} / {len(df):,} ({100*n_match/len(df):.1f}%)")
print(f"Pairs with at least one ligand diff: {len(df) - n_match:,}")
print(f"Median |R-free cryo - R-free amb|:   {df['r_free_diff_abs'].median():.3f}")

print("\nMost common diff-sets among mismatched pairs:")
print(df.loc[~df["ligand_match"], "ligands_diff_set"].value_counts().head(15))

df[["pair_id", "ligands_cryo", "ligands_ambient", "ligand_match",
    "ligands_diff_set", "r_free_cryo", "r_free_ambient"]].head(10)

Pairs with matching ligand state:    299 / 838 (35.7%)
Pairs with at least one ligand diff: 539
Median |R-free cryo - R-free amb|:   0.024

Most common diff-sets among mismatched pairs:
ligands_diff_set
MG         10
CA          8
BGC         4
ZN          4
CU,CU1      4
GLC         3
MLI         3
MN          3
CMO,PER     2
LYS         2
ACI         2
HDF         2
FE,MN       2
NAG         2
CD          2
Name: count, dtype: int64


,pair_id,ligands_cryo,ligands_ambient,ligand_match,ligands_diff_set,r_free_cryo,r_free_ambient
0,P94692_1KEK_2C3P,"CA,CO2,HTL,MG,SF4","1TP,CA,MG,SF4",False,"1TP,CO2,HTL",0.22700,0.22300
1,P22983_2DIK_1DIK,,,True,,NaN,NaN
2,Q54468_1C7T_1QBA,,,True,,0.24600,0.19600
3,Q56F26_2VZS_2X09,"CD,GCS","CD,X09",False,"GCS,X09",0.22000,0.24032
4,P09186_1ROV_1N8Q,FE,"DHB,FE2",False,"DHB,FE,FE2",0.23100,0.24500
5,P07374_4GY7_4H9M,"ACN,NI","HAE,NI",False,"ACN,HAE",0.16470,0.19100
6,P08170_1FGQ_9O4S,FE,FE,True,,0.21580,0.21020
7,P00489_3L79_7ONF,DKX,"IMP,PLP,VKK",False,"DKX,IMP,PLP,VKK",0.20800,0.16112
8,Q6R2Q7_3AI7_7C8H,"CA,TPP","CA,LMR,SIN,TPP",False,"LMR,SIN",0.20552,0.21760
9,G0SDW4_5BJS_5VK3,ZN,ZN,True,,0.22370,0.21400


## 5. Sequence reclustering — MMseqs2 (with Jaccard fallback)

MMseqs2 `easy-cluster` at 30% identity / 80% coverage is the gold-standard quick homology cluster. We try `mmseqs` from PATH first; if it's not installed we fall back to a tighter k-mer Jaccard than notebook 02 used.

To install MMseqs2 (recommended):
- conda: `conda install -n <env> -c bioconda mmseqs2`
- Linux/Mac binary: https://github.com/soedinglab/MMseqs2/releases
- Windows: easiest path is WSL or conda inside WSL.

In [5]:
def have_mmseqs():
    return shutil.which("mmseqs") is not None


def read_fasta(path):
    s = []
    with path.open() as f:
        for line in f:
            if not line.startswith(">"):
                s.append(line.strip())
    return "".join(s)


def write_combined_fasta(uniprot_to_seq, out_path):
    with open(out_path, "w") as f:
        for up, seq in uniprot_to_seq.items():
            if seq:
                f.write(f">{up}\n{seq}\n")


def run_mmseqs2(uniprot_to_seq, min_id, cov, cov_mode):
    with tempfile.TemporaryDirectory() as tmp:
        tmp = Path(tmp)
        fasta = tmp / "all.fasta"
        write_combined_fasta(uniprot_to_seq, fasta)
        out_prefix = tmp / "result"
        cmd = ["mmseqs", "easy-cluster", str(fasta), str(out_prefix),
               str(tmp / "tmpdir"),
               "--min-seq-id", str(min_id),
               "-c", str(cov),
               "--cov-mode", str(cov_mode),
               "-v", "1"]
        subprocess.run(cmd, check=True, capture_output=True, text=True)
        cluster_tsv = Path(str(out_prefix) + "_cluster.tsv")
        rep_to_members = defaultdict(list)
        with cluster_tsv.open() as f:
            for line in f:
                rep, member = line.rstrip("\n").split("\t")
                rep_to_members[rep].append(member)
        member_to_cluster = {}
        for idx, rep in enumerate(sorted(rep_to_members)):
            cname = f"mmseqs_{idx:04d}"
            for m in rep_to_members[rep]:
                member_to_cluster[m] = cname
        return member_to_cluster


# Build UniProt -> canonical sequence map from the FASTA files written by notebook 01
uniprots_in_use = sorted(df["uniprot"].dropna().unique())
uniprot_to_seq = {}
missing_fasta = []
for up in uniprots_in_use:
    p = SEQ_DIR / f"{up}.fasta"
    if p.exists():
        seq = read_fasta(p)
        if seq:
            uniprot_to_seq[up] = seq
    else:
        missing_fasta.append(up)
print(f"Loaded sequences for {len(uniprot_to_seq):,} / {len(uniprots_in_use):,} UniProts")
if missing_fasta:
    print(f"  ! {len(missing_fasta)} UniProts missing FASTA (will be singleton clusters): "
          f"{missing_fasta[:5]}{' ...' if len(missing_fasta) > 5 else ''}")

new_cluster_map = None
cluster_method = None

if have_mmseqs():
    print("\nMMseqs2 detected. Running easy-cluster ...")
    try:
        new_cluster_map = run_mmseqs2(uniprot_to_seq,
                                      MMSEQS_MIN_SEQ_ID, MMSEQS_COVERAGE, MMSEQS_COV_MODE)
        cluster_method = f"mmseqs2_id{MMSEQS_MIN_SEQ_ID}_cov{MMSEQS_COVERAGE}"
        print(f"MMseqs2 produced {len(set(new_cluster_map.values())):,} clusters "
              f"from {len(new_cluster_map):,} sequences")
    except Exception as e:
        print(f"MMseqs2 failed ({type(e).__name__}: {e}). Will fall back to Jaccard.")
        new_cluster_map = None
else:
    print("MMseqs2 not in PATH. Falling back to k-mer Jaccard reclustering.")

Loaded sequences for 838 / 838 UniProts

MMseqs2 detected. Running easy-cluster ...
MMseqs2 failed (CalledProcessError: Command '['mmseqs', 'easy-cluster', 'C:\\Users\\JENITH~1\\AppData\\Local\\Temp\\tmpp51_1s3l\\all.fasta', 'C:\\Users\\JENITH~1\\AppData\\Local\\Temp\\tmpp51_1s3l\\result', 'C:\\Users\\JENITH~1\\AppData\\Local\\Temp\\tmpp51_1s3l\\tmpdir', '--min-seq-id', '0.3', '-c', '0.8', '--cov-mode', '0', '-v', '1']' returned non-zero exit status 1.). Will fall back to Jaccard.


## 6. Fallback — k-mer Jaccard reclustering (only runs if MMseqs2 unavailable)

Tighter than notebook 02's defaults: k=4 (more discriminating) at threshold 0.10 (more aggressive merging). Roughly catches homologs around 30–40% identity. Still inferior to MMseqs2 — install MMseqs2 if at all possible for the final submission.

In [6]:
def kmers(seq, k):
    return {seq[i:i + k] for i in range(len(seq) - k + 1)} if len(seq) >= k else set()


def jaccard(a, b):
    if not a or not b:
        return 0.0
    return len(a & b) / len(a | b)


def jaccard_cluster(uniprot_to_seq, k, threshold):
    ups = sorted(uniprot_to_seq)
    n = len(ups)
    parent = list(range(n))
    def find(x):
        while parent[x] != x:
            parent[x] = parent[parent[x]]
            x = parent[x]
        return x
    def union(a, b):
        ra, rb = find(a), find(b)
        if ra != rb:
            parent[rb] = ra
    kmer_sets = [kmers(uniprot_to_seq[u], k) for u in ups]
    for i in tqdm(range(n), desc=f"Jaccard k={k} t={threshold}"):
        for j in range(i + 1, n):
            if jaccard(kmer_sets[i], kmer_sets[j]) >= threshold:
                union(i, j)
    raw = [find(i) for i in range(n)]
    cmap = {r: f"jacc_{idx:04d}" for idx, r in enumerate(sorted(set(raw)))}
    return {ups[i]: cmap[raw[i]] for i in range(n)}


if new_cluster_map is None:
    new_cluster_map = jaccard_cluster(uniprot_to_seq, JACCARD_K, JACCARD_THRESHOLD)
    cluster_method = f"jaccard_k{JACCARD_K}_t{JACCARD_THRESHOLD}"
    print(f"Jaccard produced {len(set(new_cluster_map.values())):,} clusters "
          f"from {len(new_cluster_map):,} sequences")

print(f"\nCluster method used: {cluster_method}")

Jaccard k=4 t=0.1: 100%|██████████| 838/838 [00:10<00:00, 80.29it/s] 

Jaccard produced 769 clusters from 838 sequences

Cluster method used: jaccard_k4_t0.1


## 7. Attach new cluster IDs to the manifest

In [7]:
df["seq_cluster_v3"] = df["uniprot"].map(new_cluster_map)
n_missing = df["seq_cluster_v3"].isna().sum()
if n_missing:
    # UniProts with no FASTA -> give each its own singleton cluster
    df.loc[df["seq_cluster_v3"].isna(), "seq_cluster_v3"] = (
        "singleton_" + df.loc[df["seq_cluster_v3"].isna(), "uniprot"].astype(str))
    print(f"Note: {n_missing} rows had no FASTA — assigned singleton clusters")

print(f"Old clusters (notebook 02): {df['seq_cluster'].nunique():>5,}")
print(f"New clusters (v3):          {df['seq_cluster_v3'].nunique():>5,}")
print(f"  -> reduction: {df['seq_cluster'].nunique() - df['seq_cluster_v3'].nunique()} clusters merged")

# Show the largest new clusters
top = df.groupby("seq_cluster_v3")["uniprot"].nunique().sort_values(ascending=False).head(10)
print("\nLargest v3 clusters (n_uniprots):")
print(top)

Old clusters (notebook 02):   809
New clusters (v3):            769
  -> reduction: 40 clusters merged

Largest v3 clusters (n_uniprots):
seq_cluster_v3
jacc_0212    5
jacc_0172    4
jacc_0118    4
jacc_0297    4
jacc_0252    3
jacc_0132    3
jacc_0022    3
jacc_0000    3
jacc_0193    3
jacc_0073    3
Name: uniprot, dtype: int64


## 8. Re-run cluster-aware train/val/test split

Same logic as notebook 02 — shuffle clusters, slice 70/15/15, assign each pair its cluster's split. Deterministic with `SPLIT_SEED`.

In [8]:
assert abs(TRAIN_FRAC + VAL_FRAC + TEST_FRAC - 1.0) < 1e-9
rng = random.Random(SPLIT_SEED)

clusters = sorted(df["seq_cluster_v3"].unique())
rng.shuffle(clusters)

n_clusters = len(clusters)
n_train = int(TRAIN_FRAC * n_clusters)
n_val   = int(VAL_FRAC   * n_clusters)
train_set = set(clusters[:n_train])
val_set   = set(clusters[n_train:n_train + n_val])
test_set  = set(clusters[n_train + n_val:])

def assign_split(c):
    if c in train_set: return "train"
    if c in val_set:   return "val"
    return "test"

df["split_v3"] = df["seq_cluster_v3"].apply(assign_split)
print("Split distribution (v3):")
print(df["split_v3"].value_counts().to_dict())

split_summary = (df.groupby("split_v3")
                   .agg(n_pairs=("pair_id", "count"),
                        n_uniprots=("uniprot", "nunique"),
                        n_clusters=("seq_cluster_v3", "nunique"),
                        n_modeled_both=("n_modeled_both", "sum"),
                        n_only_ambient=("n_only_ambient", "sum"))
                   .reset_index())
split_summary

Split distribution (v3):
{'train': 587, 'test': 126, 'val': 125}


,split_v3,n_pairs,n_uniprots,n_clusters,n_modeled_both,n_only_ambient
0,test,126,126,116,34383,140
1,train,587,587,538,171436,1103
2,val,125,125,115,31626,184


## 9. Leakage sanity check

No UniProt or v3 cluster should appear in more than one split.

In [9]:
splits = ("train", "val", "test")
ok = True
for i, a in enumerate(splits):
    for b in splits[i + 1:]:
        a_ups = set(df.loc[df["split_v3"] == a, "uniprot"])
        b_ups = set(df.loc[df["split_v3"] == b, "uniprot"])
        a_cl  = set(df.loc[df["split_v3"] == a, "seq_cluster_v3"])
        b_cl  = set(df.loc[df["split_v3"] == b, "seq_cluster_v3"])
        shared_ups = a_ups & b_ups
        shared_cl  = a_cl & b_cl
        if shared_ups or shared_cl:
            ok = False
        print(f"  {a:5s} <-> {b:5s}: "
              f"shared UniProts = {len(shared_ups):>3d}, "
              f"shared clusters = {len(shared_cl):>3d}")
print("\nLeakage clean:", ok)
assert ok, "Leakage detected — check cluster assignment"

  train <-> val  : shared UniProts =   0, shared clusters =   0
  train <-> test : shared UniProts =   0, shared clusters =   0
  val   <-> test : shared UniProts =   0, shared clusters =   0

Leakage clean: True


## 10. Write final manifests

`manifest_v3.csv` keeps every surviving pair (with `ligand_match` flag so you can filter at training time).
`manifest_v3_strict.csv` keeps only `ligand_match=True` rows — the conservative, publication-quality subset.

In [10]:
df.to_csv(OUTPUT_FULL_MANIFEST, index=False)
print(f"Wrote {len(df):>4,} rows -> {OUTPUT_FULL_MANIFEST.name}")

strict = df.loc[df["ligand_match"]].copy().reset_index(drop=True)
strict.to_csv(OUTPUT_STRICT_MANIFEST, index=False)
print(f"Wrote {len(strict):>4,} rows -> {OUTPUT_STRICT_MANIFEST.name}")

print("\nStrict subset split distribution:")
print(strict["split_v3"].value_counts().to_dict())
print("\nStrict subset cluster counts per split:")
print(strict.groupby("split_v3")["seq_cluster_v3"].nunique().to_dict())

Wrote  838 rows -> manifest_v3.csv
Wrote  299 rows -> manifest_v3_strict.csv

Strict subset split distribution:
{'train': 208, 'test': 47, 'val': 44}

Strict subset cluster counts per split:
{'test': 42, 'train': 199, 'val': 43}


## 11. Before / after summary

For your Methods section.

In [11]:
summary_rows = [
    ("input_pairs (from notebook 02)",       len(manifest)),
    ("pairs_v3_full",                        len(df)),
    ("pairs_v3_strict_ligand_match",         int(df["ligand_match"].sum())),
    ("pairs_dropped_for_ligand_mismatch",    int((~df["ligand_match"]).sum())),
    ("uniprots_v3",                          int(df["uniprot"].nunique())),
    ("clusters_v2 (notebook 02)",            int(df["seq_cluster"].nunique())),
    ("clusters_v3 (this notebook)",          int(df["seq_cluster_v3"].nunique())),
    ("cluster_method_v3",                    cluster_method),
    ("median_r_free_diff_abs",               float(df["r_free_diff_abs"].median())
                                              if df["r_free_diff_abs"].notna().any() else None),
    ("median_b_iso_mean_diff",               float((df["b_iso_mean_ambient"] - df["b_iso_mean_cryo"]).abs().median())
                                              if df["b_iso_mean_cryo"].notna().any() else None),
    ("split_v3_train_pairs",                 int((df["split_v3"] == "train").sum())),
    ("split_v3_val_pairs",                   int((df["split_v3"] == "val").sum())),
    ("split_v3_test_pairs",                  int((df["split_v3"] == "test").sum())),
]
summary_df = pd.DataFrame(summary_rows, columns=["Metric", "Value"])
summary_df.to_csv(SUMMARY_CSV, index=False)
print(summary_df.to_string(index=False))
print(f"\nWrote summary to {SUMMARY_CSV.name}")

                           Metric           Value
   input_pairs (from notebook 02)             838
                    pairs_v3_full             838
     pairs_v3_strict_ligand_match             299
pairs_dropped_for_ligand_mismatch             539
                      uniprots_v3             838
        clusters_v2 (notebook 02)             809
      clusters_v3 (this notebook)             769
                cluster_method_v3 jaccard_k4_t0.1
           median_r_free_diff_abs          0.0238
           median_b_iso_mean_diff          6.0074
             split_v3_train_pairs             587
               split_v3_val_pairs             125
              split_v3_test_pairs             126

Wrote summary to qc_v3_summary.csv


## Next steps

- For training, decide whether to use the **full** manifest (`manifest_v3.csv`, includes ligand-mismatched pairs — you can downweight them or mask binding-site residues) or the **strict** manifest (`manifest_v3_strict.csv`, drops them entirely). The strict version is what reviewers will expect to see in the headline numbers.
- The new `split_v3` column is the homology-aware split. Use that, not the v2 `split` column.
- AdoMetDC (P17707, 9P1H/9P7Q/9PBB) is still excluded — keep it that way for the held-out case study.
- For the writeup, the `cluster_method_v3` string in `qc_v3_summary.csv` documents which clustering produced the splits — useful in the Methods section.